# SPMD across a multi-host slice — hands-on

A companion notebook for the lesson [*SPMD — One Program Across Every Host*](https://lms-p-45c03.web.app/topics/ml-systems/spmd-multi-host/).

You'll **run the SPMD model yourself**: build a device mesh, shard a tensor with a
`PartitionSpec`, *see* who holds which shard, watch the compiler **insert the collectives**
you never wrote, and compare automatic sharding with manual `shard_map`.

### Fake devices vs real ones — does it matter?
For everything here, **no** — that's JAX's whole promise: the same program runs on any backend.
The mesh, the shardings, the `visualize_array_sharding` picture, and *which collective* the
compiler inserts are **identical** on 8 fake CPU devices and 8 real TPU chips. What differs is
only **timing** (fake devices time-slice one CPU — there's no real interconnect, so a benchmark
measures nothing) and the device *kind* shown.

**The one thing neither option exercises: true multi-*host*.** `XLA_FLAGS` fakes 8 *devices in one
process*, and a single 8-chip TPU VM is also one host. The `jax.distributed.initialize` /
coordinator / DCN rendezvous — the heart of the lesson's failure story — needs **≥2 host VMs**.
This notebook is single-process multi-device; the last cell *shows* the multi-host API and explains
why it can't run here.

> Runs on the default **CPU** runtime (8 fake devices). For real chips: `Runtime → Change runtime
> type → a TPU with ≥8 chips`, then set `USE_FAKE_DEVICES = False`. Version-robust via shims;
> verified on JAX 0.4.x, written for 0.7.x. Run top to bottom.

## 1. Get your devices
Set the toggle. On a TPU runtime with ≥8 chips, flip it to `False`.

In [ ]:
USE_FAKE_DEVICES = True   # 8 fake CPU devices — runs anywhere. False on a TPU runtime with >=8 chips.

import os
if USE_FAKE_DEVICES:
    # MUST be set before importing jax — fakes 8 devices in this one process.
    os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"

import numpy as np
import jax, jax.numpy as jnp
from jax.sharding import PartitionSpec as P, NamedSharding

print("JAX", jax.__version__, "|", len(jax.devices()), "x", jax.devices()[0].platform.upper())
assert len(jax.devices()) >= 8, "Need >=8 devices: pick a >=8-chip TPU runtime, or keep USE_FAKE_DEVICES=True."

## Across JAX versions
`make_mesh` and `shard_map` became top-level in newer JAX (they were elsewhere in 0.4.x). These
shims make the rest of the notebook run on either.

In [ ]:
def make_mesh(shape, axes):
    if hasattr(jax, "make_mesh"):
        return jax.make_mesh(shape, axes)                       # newer JAX
    return jax.sharding.Mesh(np.array(jax.devices()[:int(np.prod(shape))]).reshape(shape), axes)

try:
    from jax import shard_map                                   # newer JAX (top-level)
except ImportError:
    from jax.experimental.shard_map import shard_map

## 2. See the sharding
Build a 2x4 mesh (`data`=2, `model`=4) and shard a `[batch, features]` tensor three ways. The
ASCII map shows exactly which device holds which shard — the same three layouts the lesson's
explorer draws. (`visualize_array_sharding` needs `rich`, which Colab has pre-installed.)

In [ ]:
mesh = make_mesh((2, 4), ('data', 'model'))     # data=2, model=4  -> 8 devices
batch, features = 512, 1024
x = jnp.arange(batch * features, dtype=jnp.bfloat16).reshape(batch, features)

for name, spec in [("P('data', None)  - shard the batch (data-parallel)", P('data', None)),
                   ("P(None, 'model')  - shard the features (model-parallel)", P(None, 'model')),
                   ("P()               - fully replicated", P())]:
    xs = jax.device_put(x, NamedSharding(mesh, spec))
    print("\n" + name)
    jax.debug.visualize_array_sharding(xs)

## 3. Watch the compiler insert the collective
You declared a layout — you never wrote a `send`/`recv`. Reduce across the sharded `data` axis,
then read the **compiled HLO**: Shardy put an `all-reduce` in for you (same `lower().compile()`
muscle as the 2.1 notebook).

In [ ]:
xs = jax.device_put(x, NamedSharding(mesh, P('data', None)))   # data-parallel layout

@jax.jit
def reduce_over_batch(a):
    return jnp.sum(a, axis=0)                 # a cross-shard reduction over the data axis

hlo = jax.jit(reduce_over_batch).lower(xs).compile().as_text()
hits = [l.strip() for l in hlo.splitlines()
        if any(k in l for k in ('all-reduce', 'reduce-scatter', 'all-gather'))]
print("Collectives Shardy inserted (you wrote none):")
for h in hits[:3]:
    print("  ", h[:90])

## 4. Auto vs manual (`shard_map`)
The same reduction two ways: **Auto** lets Shardy insert the collective; **Manual** (`shard_map`)
makes you write it (`jax.lax.psum`). Same answer, different control — this is the Auto -> Explicit
-> Manual spectrum from the lesson. (Cast to float32 so bf16 reduction-order rounding doesn't muddy
the equality check.)

In [ ]:
xf = x.astype(jnp.float32)
xs2 = jax.device_put(xf, NamedSharding(mesh, P('data', 'model')))

auto = jnp.sum(xs2)                           # Auto: Shardy inserts the all-reduce

@jax.jit
def manual(a):
    f = shard_map(lambda s: jax.lax.psum(jnp.sum(s), ('data', 'model')),
                  mesh=mesh, in_specs=P('data', 'model'), out_specs=P())
    return f(a)                              # Manual: you wrote the psum
man = manual(xs2)

print("auto   =", float(auto))
print("manual =", float(man), " (shard_map + psum)")
print("match  :", bool(jnp.allclose(auto, man)))

## 5. What it costs
Per-chip memory and the collective for each layout — the numbers the explorer shows.

In [ ]:
TENSOR_MB = batch * features * 2 / (1024 * 1024)    # bf16 = 2 bytes, in MiB
print(f"Tensor [{batch}, {features}] bf16 = {TENSOR_MB:.2f} MB\n")
print(f"  P('data', None)   per-chip {TENSOR_MB/2:.2f} MB (4 copies)  -> all-reduce on the backward pass")
print(f"  P(None, 'model')  per-chip {TENSOR_MB/4:.2f} MB (2 copies)  -> all-gather to use the full activation")
print(f"  P()               per-chip {TENSOR_MB:.2f} MB (8 copies)  -> no collective, no memory saving")

## 6. The part this notebook can't run: multi-host
Everything above is one process driving 8 devices. A **real multi-host job** runs this *same file*
on every host; `jax.distributed.initialize()` points them all at a coordinator (process 0) and
blocks at a barrier over the **DCN** until all check in. You can't do that from one Colab — so the
code below is **for reference, not to run.**

In [ ]:
# DO NOT RUN here (single host). On a real multi-host job, every host runs this file:
#
#   jax.distributed.initialize()      # no args on Cloud TPU; auto-detects coordinator + process_id
#   mesh = make_mesh((n_hosts, devices_per_host), ('data', 'model'))
#   # ... the exact sharding/code above, now spanning hosts ...
#
# The #1 failure is a BARRIER TIMEOUT at startup: a host can't reach the coordinator
# (firewall, wrong IP, MTU) -- a DCN networking bug, not a TPU fault. The chips are idle;
# nothing compiled wrong. That is the failure mode the lesson tells you to debug.
print("Multi-host rendezvous needs >=2 host VMs -- see the lesson's 'Rendezvous is a networking step'.")

## Put it together

1. **Read the maps (§2):** for `P('data', None)`, which devices hold the *same* shard, and why? (Hint: the replicated axis.)
2. **Find the collective (§3):** you wrote `jnp.sum`. What did Shardy add, and over which axis — and which network would it ride on a single slice (ICI) vs across slices (DCN)?
3. **Auto vs manual (§4):** the answers match. What did `shard_map` make you write that Auto did for free — and when is that control worth it?
4. **Cost (§5):** `P(None, 'model')` uses the least memory per chip but needs an all-gather to use the tensor. State the trade in one sentence.
5. **(Multi-host)** Why can't this notebook trigger a rendezvous timeout — and what would you need to set up to see one?

Back to the lesson -> [SPMD — One Program Across Every Host](https://lms-p-45c03.web.app/topics/ml-systems/spmd-multi-host/)